# Part 0: Setup & Data Loading

Imports, paths, constants, utility functions, and initial data loading.

Run this notebook first. It saves intermediate state for all downstream parts.

In [1]:
import os, shutil, re
from collections import Counter
import numpy as np
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns

# ── Paths ─────────────────────────────────────────────────────────────────

BASE_DIR = "/Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile"
DIR_PRELIM = os.path.join(BASE_DIR, "Preliminary")
COCHILCO_PATH = os.path.join(BASE_DIR, "data", "COCHILCO_Production_2005_2024.xlsx")

_cochilco_orig_candidates = [
    os.path.join(BASE_DIR, "data", "1771263160312_Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx"),
    os.path.join(BASE_DIR, "data", "Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx"),
    "/Users/leoss/Downloads/1771263160312_Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx",
    "/Users/leoss/Downloads/Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx",
]
COCHILCO_ORIG = next((p for p in _cochilco_orig_candidates if os.path.exists(p)), _cochilco_orig_candidates[0])

# ── Shared constants ──────────────────────────────────────────────────────
# COMPANY_TO_DEPOSIT: COCHILCO company -> inventory search terms
# Codelco divisions include extra search terms for associated facilities.

COMPANY_TO_DEPOSIT = {
    "División El Teniente": ["El Teniente", "Teniente"],
    "División Chuquicamata": ["Chuquicamata"],
    "División Radomiro Tomic": ["Radomiro Tomic", "Radomiro"],
    "División Andina": ["Andina"],
    "División Ministro Hales": ["Ministro Hales"],
    "División Gabriela Mistral": ["Gabriela Mistral", "Gaby"],
    "División Salvador": ["Salvador"],
    "Escondida": ["Escondida"], "Collahuasi": ["Collahuasi"],
    "Los Pelambres": ["Los Pelambres", "Pelambres"],
    "Spence": ["Spence"], "Quebrada Blanca": ["Quebrada Blanca"],
    "Los Bronces": ["Los Bronces", "Bronces"],
    "Sierra Gorda": ["Sierra Gorda"], "Candelaria": ["Candelaria"],
    "Caserones": ["Caserones"], "Centinela (Súlfuros)": ["Centinela"],
    "El Abra": ["El Abra", "Abra"], "Zaldívar": ["Zaldívar", "Zaldivar"],
    "Antucoya": ["Antucoya"], "Lomas Bayas": ["Lomas Bayas"],
    "Mantoverde": ["Mantoverde"], "El Soldado": ["El Soldado", "Soldado"],
    "Mantos Blancos": ["Mantos Blancos"], "Andacollo": ["Carmen de Andacollo"],  # not Andacollo Oro (gold)
    "Michilla": ["Michilla"], "Franke": ["Franke"],
    "Mantos de la Luna": ["Mantos de la Luna"],
    "Cerro Negro": ["Cerro Negro"],
    "Las Cenizas": ["Las Cenizas", "Cenizas", "Las Luces", "Aguilucho"],
    "Tres Valles": ["Tres Valles"],
}

# Extended search terms for Codelco divisions (includes non-mine facilities)
CODELCO_EXTRA_SEARCH = {
    "División Andina": ["Rio Blanco"],
    "División Ministro Hales": ["Mansa Mina"],
    "División Salvador": ["El Salvador", "Potrerillos"],
}

SMELTERS = [
    {"name": "Chuquicamata smelter",
     "search": ["Chuquicamata SX-EW plant (oxide) and smelter", "Chuquicamata Division", "Chuquicamata plant"],
     "operator": "Codelco", "smelter_type": "integrated", "region": "Antofagasta",
     "lat": -22.32, "lon": -68.93, "has_refinery": True,
     "feeds_from_mines": ["Chuquicamata", "Radomiro Tomic", "Ministro Hales"],
     "output_product": "cathode", "export_ports": ["Angamos", "Mejillones"]},
    {"name": "Potrerillos smelter",
     "search": ["Potrerillos SX-EW refinery and smelter", "Potrerillos plant"],
     "operator": "Codelco", "smelter_type": "integrated", "region": "Atacama",
     "lat": -26.39, "lon": -69.46, "has_refinery": True,
     "feeds_from_mines": ["Salvador"],
     "output_product": "cathode", "export_ports": ["Barquito"]},
    {"name": "Caletones smelter",
     "search": ["El Teniente plant"],
     "operator": "Codelco", "smelter_type": "integrated", "region": "O'Higgins",
     "lat": -34.12, "lon": -70.48, "has_refinery": True,
     "feeds_from_mines": ["El Teniente"],
     "output_product": "cathode", "export_ports": ["San Antonio", "Ventanas"]},
    {"name": "Altonorte smelter",
     "search": ["Altonorte"],
     "operator": "Glencore", "smelter_type": "custom", "region": "Antofagasta",
     "lat": -23.78, "lon": -70.31, "has_refinery": False,
     "feeds_from_mines": [], "feeds_from_region": "Antofagasta",
     "output_product": "blister", "export_ports": ["Antofagasta", "Mejillones"]},
    {"name": "Paipote smelter (H.V. Lira)",
     "search": ["Paipote", "Hernan Videla", "Hernán Videla"],
     "operator": "ENAMI", "smelter_type": "custom", "region": "Atacama",
     "lat": -27.37, "lon": -70.30, "has_refinery": False,
     "feeds_from_mines": [], "feeds_from_region": "Atacama",
     "output_product": "blister", "export_ports": ["Barquito", "Caldera"]},
    {"name": "Chagres smelter",
     "search": ["Chagres"],
     "operator": "Anglo American", "smelter_type": "integrated", "region": "Valparaiso",
     "lat": -32.78, "lon": -70.97, "has_refinery": False,
     "feeds_from_mines": ["Los Bronces", "El Soldado"],
     "output_product": "blister", "export_ports": ["Ventanas"]},
]

PORTS = [
    {"name": "Coloso",            "region": "Antofagasta", "lat": -23.76, "lon": -70.45,
     "products": "Cu concentrate", "key_users": "Escondida (dedicated)"},
    {"name": "Angamos",           "region": "Antofagasta", "lat": -23.10, "lon": -70.42,
     "products": "Cu cathode, concentrate", "key_users": "Codelco"},
    {"name": "Mejillones",        "region": "Antofagasta", "lat": -23.10, "lon": -70.45,
     "products": "Cu concentrate, cathode, acid", "key_users": "Multiple"},
    {"name": "Antofagasta (ATI)", "region": "Antofagasta", "lat": -23.65, "lon": -70.40,
     "products": "Cu cathode, general", "key_users": "Antofagasta Minerals"},
    {"name": "Iquique",           "region": "Tarapaca",    "lat": -20.21, "lon": -70.15,
     "products": "Cu cathode", "key_users": "Collahuasi, Quebrada Blanca"},
    {"name": "Patache",           "region": "Tarapaca",    "lat": -20.80, "lon": -70.22,
     "products": "Cu concentrate", "key_users": "Collahuasi, Quebrada Blanca"},
    {"name": "Barquito",          "region": "Atacama",     "lat": -27.07, "lon": -70.84,
     "products": "Cu concentrate, cathode", "key_users": "Codelco Salvador, ENAMI"},
    {"name": "Caldera",           "region": "Atacama",     "lat": -27.07, "lon": -70.82,
     "products": "Cu concentrate", "key_users": "Medium miners"},
    {"name": "Coquimbo",          "region": "Coquimbo",    "lat": -29.96, "lon": -71.35,
     "products": "Cu concentrate, Fe pellets", "key_users": "CMP"},
    {"name": "Los Vilos",         "region": "Coquimbo",    "lat": -31.91, "lon": -71.51,
     "products": "Cu concentrate", "key_users": "Los Pelambres (dedicated)"},
    {"name": "Ventanas",          "region": "Valparaiso",  "lat": -32.74, "lon": -71.49,
     "products": "Cu cathode, blister", "key_users": "Codelco, Anglo American"},
    {"name": "San Antonio",       "region": "Valparaiso",  "lat": -33.59, "lon": -71.62,
     "products": "Cu cathode, general", "key_users": "El Teniente, general"},
    {"name": "San Vicente",       "region": "Biobio",      "lat": -36.73, "lon": -73.13,
     "products": "Cu cathode, general", "key_users": "Southern region"},
]

SMELTER_NAME_MAP = {
    "Chuquicamata smelter":        "Chuquicamata SX-EW plant (oxide) and smelter",
    "Potrerillos smelter":         "Potrerillos SX-EW refinery and smelter",
    "Caletones smelter":           "Caletones smelter (anodes). refinery (fire-refined ingots), and SX-EW plant",
    "Altonorte smelter":           "Altonorte smelter",
    "Paipote smelter (H.V. Lira)": "Hernán Videla Lira smelter (anodes and blister)",
    "Chagres smelter":             "Chagres smelter (anodes and blister)",
}

# Mine -> dedicated port overrides (contractual, not distance-based)
# These override the nearest_port() logic in downstream edge construction.
DEDICATED_PORT = {
    "Escondida": "Coloso",
    "Los Pelambres": "Los Vilos",
    "Pelambres": "Los Vilos",
}

# Codelco cathode consolidation: Codelco divisions ship cathode through Angamos
# regardless of geographic proximity, except Salvador which uses Barquito,
# and El Teniente/Andina which use San Antonio.
# When substring matching returns >1 mine, prefer this exact name.
MATCH_DISAMBIGUATION = {
    "Cerro Negro": "Cerro Negro",       # not "Cerro Negro Norte" (iron)
    "Andacollo": "Carmen de Andacollo",  # not "Andacollo Oro" (gold)
}

# Known iron mine operators for weighted Fe assignment in Part 2.
IRON_MINE_NAMES = [
    "Los Colorados", "El Algarrobo", "Cerro Negro Norte",
    "Romeral", "Tofo", "Distrito El Algarrobo",
]

# Known zinc mine for direct assignment.
ZINC_MINE_NAMES = ["El Toqui"]

CODELCO_CATHODE_ROUTING = {
    "Chuquicamata": "Angamos",
    "Radomiro Tomic": "Angamos",
    "Ministro Hales": "Angamos",
    "Gabriela Mistral": "Angamos",
    "Salvador": "Barquito",
    "El Teniente": "San Antonio",
    "Andina": "San Antonio",
}

# ── Utility functions ─────────────────────────────────────────────────────

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def parse_comm_list(val):
    if pd.isna(val): return []
    return [x.strip() for x in str(val).split(",") if x.strip()]

def add_commodity(row_idx, commodity, df, col):
    current = parse_comm_list(df.at[row_idx, col])
    if commodity not in current:
        current.append(commodity)
        df.at[row_idx, col] = ", ".join(current)
        return True
    return False

def nearest_port(lat, lon, product_type="concentrate"):
    best_dist, best_port = float("inf"), None
    for port in PORTS:
        if product_type == "cathode" and "cathode" not in port["products"].lower():
            continue
        if product_type == "concentrate" and "concentrate" not in port["products"].lower():
            continue
        dist = haversine_km(lat, lon, port["lat"], port["lon"])
        if dist < best_dist:
            best_dist, best_port = dist, port
    return best_port, best_dist

def section_header(title, width=65):
    print(f"\n{'=' * width}\n{title}\n{'=' * width}")

def search_inventory(inv_df, terms, require_mine=False):
    """Search inventory by a list of terms. Returns matching indices."""
    matched = set()
    name_lower = inv_df["FACILITY_NAME"].str.lower().str.strip()
    for term in terms:
        mask = name_lower.str.contains(term.lower(), na=False, regex=False)
        if require_mine:
            mask = mask & inv_df["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)
        matched.update(inv_df[mask].index)
    return sorted(matched)  # deterministic ordering across runs

# ── Load data (with idempotent backup/reload) ─────────────────────────────

inv_path = os.path.join(DIR_PRELIM, "Chile_Minerals_Inventory.csv")
links_path = os.path.join(DIR_PRELIM, "Chile_Mine_Plant_Links.csv")

# Always reload from backup if present (guarantees clean state on re-run)
for p, label in [(inv_path, "inventory"), (links_path, "links")]:
    bak = p.replace(".csv", "_backup.csv")
    if not os.path.exists(bak):
        shutil.copy2(p, bak)
        print(f"Backed up: {bak}")

inv = pd.read_csv(inv_path.replace(".csv", "_backup.csv") if os.path.exists(inv_path.replace(".csv", "_backup.csv")) else inv_path, low_memory=False)
links = pd.read_csv(links_path.replace(".csv", "_backup.csv") if os.path.exists(links_path.replace(".csv", "_backup.csv")) else links_path)

comm_col = "COMMODITY_LIST_STR" if "COMMODITY_LIST_STR" in inv.columns else "ALL_COMMODITIES_RAW"

print(f"Inventory: {len(inv)} records")
print(f"Links: {len(links)} records")

# Remove idle mines from supply chain edges (keep in inventory for context)
idle_mines = set(inv.loc[inv["FACILITY_TYPE"] == "Mine (idle)", "FACILITY_NAME"])
idle_link_mask = links["MINE_NAME"].isin(idle_mines)
n_idle_links = idle_link_mask.sum()
links = links[~idle_link_mask].reset_index(drop=True)
print(f"Removed {n_idle_links} links from {len(idle_mines)} idle mines")
print(f"Links after idle filter: {len(links)}")



# ── Save intermediate state for downstream notebooks ──────────────────────
import pickle
_state = {
    "inv": inv, "links": links, "comm_col": comm_col, "idle_mines": idle_mines,
    "COMPANY_TO_DEPOSIT": COMPANY_TO_DEPOSIT,
    "CODELCO_EXTRA_SEARCH": CODELCO_EXTRA_SEARCH,
    "SMELTERS": SMELTERS, "PORTS": PORTS,
    "SMELTER_NAME_MAP": SMELTER_NAME_MAP,
    "DEDICATED_PORT": DEDICATED_PORT,
    "CODELCO_CATHODE_ROUTING": CODELCO_CATHODE_ROUTING,
    "MATCH_DISAMBIGUATION": MATCH_DISAMBIGUATION,
    "IRON_MINE_NAMES": IRON_MINE_NAMES,
    "ZINC_MINE_NAMES": ZINC_MINE_NAMES,
}
_state_path = os.path.join(DIR_PRELIM, "_pipeline_state_0.pkl")
with open(_state_path, "wb") as _f:
    pickle.dump(_state, _f)
print(f"State saved to {_state_path}")


Inventory: 461 records
Links: 1370 records
Removed 263 links from 80 idle mines
Links after idle filter: 1107
State saved to /Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile/Preliminary/_pipeline_state_0.pkl
